# Emoji tokenization audit — `Davlan/afro-xlmr-large`

Question: if a customer message contains emoji, does this tokenizer keep them as real pieces, or do they become `<unk>`?

The samples below are synthetic. They are not customer messages. Do not paste a real ticket into this notebook and commit the executed output.

AfroXLMR-large is an encoder-only XLM-RoBERTa. Its adaptation paper removed non-African writing-script tokens from the embedding layer before masked-language-model training (Alabi et al. 2022). Emoji were not the point of that model. This notebook still measures the tokenizer instead of assuming the outcome.

`lib/api.ts` posts the message it was given. Nothing in this repo strips emoji before `POST /predict`. The production policy from this audit is: keep emoji, do not add a strip step.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "notebooks" / "emoji_audit.py").is_file() else HERE.parent
if not (ROOT / "notebooks" / "emoji_audit.py").is_file():
    raise SystemExit(f"Run from the repo root or from notebooks/. cwd={HERE}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"repo root: {ROOT}")


## CONFIG

`samples` is the whole audit set. Add another synthetic string here if you want, and keep real customer text out.


In [ ]:
CONFIG = {
    "encoder_name": "Davlan/afro-xlmr-large",
    "samples": [
        "Please reverse the Airtel Money payment I sent to the wrong number.",
        "Please reverse the Airtel Money payment I sent to the wrong number 🙏",
        "Data bundle imeisha but si urgent — I can buy another pack tomorrow.",
        "Data bundle imeisha but si urgent 😭 I can buy another pack tomorrow.",
        "My home 5G router has a red light and WiFi is down 😡",
        "Thank you, the bundle you credited yesterday is working ✅",
        "Please bar this line immediately 🚨 I lost the handset.",
        "Family line: mama na watoto 👨‍👩‍👧 wanashare bundle.",
        "Thumb 👍🏽 and fire 🔥 on the same ticket.",
        "Kenya flag test 🇰🇪 on a synthetic line.",
    ],
}
print(f"{len(CONFIG['samples'])} synthetic samples")
print("encoder:", CONFIG["encoder_name"])


## Tokenize with and without emoji

The tokenizer download is the small SentencePiece/tokenizer file, not the full encoder weights. Each row shows the tokens for the original string, the tokens after emoji are removed, and whether the difference is `<unk>` or a real piece.


In [ ]:
from notebooks.emoji_audit import audit_text, production_policy, summarize_audits

summary = {"aggregate": "unmeasured", "counts": {}, "n": 0}
audits = None
try:
    from transformers import AutoTokenizer
except ImportError as err:
    print(f"Tokenizer library is not installed ({err}).")
    print("pip install -r notebooks/requirements.txt")
else:
    print(f"loading tokenizer {CONFIG['encoder_name']}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(CONFIG["encoder_name"])
    except OSError as err:
        print(f"Tokenizer was not downloaded: {err}")
    else:
        print(f"unk token: {tokenizer.unk_token!r} id={tokenizer.unk_token_id} vocab={tokenizer.vocab_size}")
        audits = [audit_text(tokenizer, text) for text in CONFIG["samples"]]
        summary = summarize_audits(audits)
        print(f"aggregate: {summary['aggregate']}")
        print(f"counts: {summary['counts']}")
        for item in audits:
            print()
            print(f"status={item['status']} extra_unk={item['extra_unk']} emoji={item['emoji']!r}")
            print(f"  with:    {item['tokens']}")
            print(f"  without: {item['tokens_without_emoji']}")


## Production policy

Keep emoji. The paragraph below is the measured reason, or an explicit "not measured yet" if the tokenizer did not load.


In [ ]:
print(production_policy(summary))
print()
print("This repo already behaves that way: lib/api.ts sends the message text unchanged, and eval/run.mjs sends the CSV `text` column unchanged.")
